In [0]:
%pip install databricks-sdk openai --upgrade --quiet
dbutils.library.restartPython()

# Step 3: AgentBricks Supervisor Agent

This notebook creates a **managed Supervisor Agent** using the Databricks AgentBricks SDK.

The supervisor coordinates two Genie spaces as sub-agents:

```
┌───────────────────────────────────────────────────────┐
│          MANAGED SUPERVISOR AGENT                     │
│  (AgentBricks handles routing + orchestration)        │
└───────────────────────┬───────────────────────────────┘
                        │
           ┌───────────┴───────────┐
           ▼                       ▼
┌──────────────────┐  ┌──────────────────┐
│ Genie Space:       │  │ Genie Space:       │
│ Procurement &      │  │ Logistics &        │
│ Inventory          │  │ Fulfillment        │
└──────────────────┘  └──────────────────┘
```

**What AgentBricks handles automatically:**
- Intent classification / routing between sub-agents
- Context passing between tools
- Endpoint creation for downstream apps (Playground, Apps)
- Access control (end users only see data they have permission for)

**Prerequisites:** Run notebooks 01 and 02 first.

In [0]:
from databricks.sdk import WorkspaceClient
import os

w = WorkspaceClient()

# Shared config
CATALOG = "agents"
SCHEMA = "supply_chain_demo"
SPACE_1_TITLE = "Supply Chain - Procurement & Inventory"
SPACE_2_TITLE = "Supply Chain - Logistics & Fulfillment"
SUPERVISOR_DISPLAY_NAME = "Supply Chain Supervisor"

# Dynamically find Genie space IDs by title
response = w.genie.list_spaces()
all_spaces = response.spaces or []

procurement_matches = [s for s in all_spaces if s.title == SPACE_1_TITLE]
logistics_matches = [s for s in all_spaces if s.title == SPACE_2_TITLE]

if not procurement_matches:
    raise ValueError(f"Genie space '{SPACE_1_TITLE}' not found. Run notebook 02 first.")
if not logistics_matches:
    raise ValueError(f"Genie space '{SPACE_2_TITLE}' not found. Run notebook 02 first.")

PROCUREMENT_SPACE_ID = procurement_matches[0].space_id
LOGISTICS_SPACE_ID = logistics_matches[0].space_id

# Workspace host for API calls
HOST = w.config.host

print("Configuration:")
print(f"  Catalog: {CATALOG}")
print(f"  Schema:  {SCHEMA}")
print(f"  Host:    {HOST}")
print(f"  Procurement space: {PROCUREMENT_SPACE_ID}")
print(f"  Logistics space:   {LOGISTICS_SPACE_ID}")

---
## Step 1: Create the Supervisor Agent
Uses the AgentBricks SDK to create a managed supervisor with instructions for the supply chain domain.

In [0]:
from databricks.sdk.service.supervisoragents import SupervisorAgent

# Define the supervisor agent (uses variables from config cell)
supervisor_config = SupervisorAgent(
    display_name=SUPERVISOR_DISPLAY_NAME,
    description=f"A supervisor agent that coordinates procurement/inventory and logistics/fulfillment Genie spaces to answer supply chain questions using {CATALOG}.{SCHEMA}.",
    instructions="""You are a supply chain supervisor agent. You coordinate two specialized sub-agents:

1. **Procurement & Inventory** - Handles questions about suppliers, materials, purchase orders, warehouse inventory, spend, lead times, reorder points, and stockout risk.

2. **Logistics & Fulfillment** - Handles questions about carriers, shipments, routes, delivery tracking, shipping costs, transit times, on-time rates, and delays.

When a user asks a question:
- Route procurement/inventory questions to the Procurement tool
- Route shipping/delivery questions to the Logistics tool
- For cross-domain questions (e.g., end-to-end cost from order to delivery), query both tools and synthesize the answer
- Always provide clear, actionable answers with specific data
- If a question is ambiguous, ask for clarification"""
)

# Create the supervisor
created = w.supervisor_agents.create_supervisor_agent(supervisor_agent=supervisor_config)

SUPERVISOR_ID = created.name.split("/")[-1] if hasattr(created, 'name') else str(created)
print(f"✓ Created Supervisor Agent")
print(f"  Name: {created.name}")
print(f"  Display Name: {SUPERVISOR_DISPLAY_NAME}")

---
## Step 2: Add Genie Spaces as Tools
Attach both Genie spaces as sub-agents. The supervisor will automatically route questions to the appropriate space.

In [0]:
from databricks.sdk.service.supervisoragents import Tool, GenieSpace

# Extract supervisor agent ID from the created resource name
# Format: "supervisor-agents/<id>"
agent_parent = created.name  # e.g., "supervisor-agents/abc123"

# Add Procurement & Inventory Genie Space as a tool
procurement_tool = Tool(
    tool_type="genie_space",
    description="Answers questions about procurement: suppliers, materials, purchase orders, warehouse inventory levels, spend analysis, lead times, reorder points, and stockout risk. Use for any question about buying/sourcing materials or current stock levels.",
    genie_space=GenieSpace(
        id=PROCUREMENT_SPACE_ID,
    ),
)

created_procurement = w.supervisor_agents.create_tool(
    parent=agent_parent,
    tool=procurement_tool,
    tool_id="procurement_inventory",
)
print(f"✓ Added tool: Procurement & Inventory")
print(f"  Tool ID: procurement_inventory")
print(f"  Genie Space: {PROCUREMENT_SPACE_ID}")

# Add Logistics & Fulfillment Genie Space as a tool
logistics_tool = Tool(
    tool_type="genie_space",
    description="Answers questions about logistics: carriers, shipments, delivery tracking, routes, shipping costs, transit times, on-time delivery rates, and shipment delays. Use for any question about moving/shipping goods or delivery status.",
    genie_space=GenieSpace(
        id=LOGISTICS_SPACE_ID,
    ),
)

created_logistics = w.supervisor_agents.create_tool(
    parent=agent_parent,
    tool=logistics_tool,
    tool_id="logistics_fulfillment",
)
print(f"✓ Added tool: Logistics & Fulfillment")
print(f"  Tool ID: logistics_fulfillment")
print(f"  Genie Space: {LOGISTICS_SPACE_ID}")

print(f"\n✓ Supervisor agent fully configured with 2 Genie space tools")

---
## Step 3: Verify Configuration
List the supervisor's tools to confirm both spaces are attached.

In [0]:
# List all tools on the supervisor
tools = w.supervisor_agents.list_tools(parent=agent_parent)

print("=" * 60)
print("SUPERVISOR AGENT CONFIGURATION")
print("=" * 60)
print(f"\nAgent: {agent_parent}")
print(f"Display Name: Supply Chain Supervisor")
print(f"\nTools ({len(list(tools))} sub-agents):")

# Re-list since iterator is consumed
tools = w.supervisor_agents.list_tools(parent=agent_parent)
for tool in tools:
    print(f"  • [{tool.tool_type}] {tool.name}")
    print(f"    Description: {tool.description[:80]}...")
    print()

---
## Step 4: Test in Playground
The supervisor agent is now live. You can interact with it via:
1. **AI Playground** — Search for "Supply Chain Supervisor" in the endpoint dropdown
2. **Databricks Apps** — Build a chat app pointed at the supervisor endpoint
3. **REST API** — Call the endpoint programmatically

### Example questions to try:
| Question | Expected Routing |
|----------|--|
| "Which supplier has the worst lead time?" | Procurement |
| "What's the total shipping cost by carrier?" | Logistics |
| "Are we at risk of stockout on any materials?" | Procurement |
| "How many shipments are delayed this month?" | Logistics |
| "What's our total supply chain cost (procurement + shipping)?" | Both |

In [0]:
# Derive the endpoint name from the supervisor agent ID
# Pattern: mas-<first-8-chars-of-agent-id>-endpoint
agent_id = agent_parent.split("/")[-1]
ENDPOINT_NAME = f"mas-{agent_id.split('-')[0]}-endpoint"

print("=" * 60)
print("SETUP COMPLETE")
print("=" * 60)
print(f"""
Supervisor Agent: {agent_parent}
Endpoint: {ENDPOINT_NAME}

Sub-agents:
  1. Procurement & Inventory (Genie Space: {PROCUREMENT_SPACE_ID})
  2. Logistics & Fulfillment (Genie Space: {LOGISTICS_SPACE_ID})
""")

---
## Step 5: Invoke the Supervisor Agent
Call the supervisor endpoint using the OpenAI-compatible Responses API. The supervisor routes each question to the correct Genie space automatically.

In [0]:
from openai import OpenAI
import time

# Get token from notebook context (no hardcoding)
DATABRICKS_TOKEN = dbutils.notebook.entry_point.getDbutils().notebook().getContext().apiToken().get()

client = OpenAI(
    api_key=DATABRICKS_TOKEN,
    base_url=f"{HOST}/serving-endpoints"
)

# Wait for the endpoint to become available (it takes time to provision after agent creation)
print(f"Waiting for endpoint '{ENDPOINT_NAME}' to become ready...")
max_wait_seconds = 600  # 10 minutes max
poll_interval = 15
elapsed = 0
endpoint_ready = False

while elapsed < max_wait_seconds:
    try:
        ep = w.serving_endpoints.get(ENDPOINT_NAME)
        # Compare using string representation since ep.state.ready is an enum
        if ep.state and "READY" in str(ep.state.ready):
            endpoint_ready = True
            print(f"✓ Endpoint is ready (waited {elapsed}s)")
            break
        else:
            state_msg = ep.state.ready if ep.state else "UNKNOWN"
            print(f"  Endpoint state: {state_msg} (waited {elapsed}s)...")
    except Exception as e:
        print(f"  Endpoint not found yet (waited {elapsed}s)... {e}")
    time.sleep(poll_interval)
    elapsed += poll_interval

if not endpoint_ready:
    raise TimeoutError(f"Endpoint '{ENDPOINT_NAME}' did not become ready within {max_wait_seconds}s. Check the serving endpoints UI.")

def ask_supervisor(question: str) -> str:
    """Send a question to the supervisor agent and return the response."""
    response = client.responses.create(
        model=ENDPOINT_NAME,
        input=[{"role": "user", "content": question}]
    )
    return " ".join(
        getattr(content, "text", "")
        for output in response.output
        for content in getattr(output, "content", [])
    )

# Test with sample questions
test_questions = [
    "Which supplier has the highest total spend?",
    "What carriers have the most delayed shipments?",
    "Give me a supply chain health summary across procurement and logistics",
]

for q in test_questions:
    print(f"\n{'=' * 60}")
    print(f"Q: {q}")
    print("-" * 60)
    answer = ask_supervisor(q)
    print(f"A: {answer[:500]}{'...' if len(answer) > 500 else ''}")
    print()

---
## (Optional) Cleanup
Uncomment to delete the supervisor agent and its tools.

In [0]:
# Uncomment to delete the supervisor agent:
# w.supervisor_agents.delete_supervisor_agent(name=agent_parent)
# print(f"Deleted supervisor: {agent_parent}")

---
## Architecture Summary

| Component | What it does | Managed by |
|-----------|-------------|------------|
| **Supervisor Agent** | Routes questions, synthesizes cross-domain answers | AgentBricks |
| **Procurement Tool** | Genie space for suppliers, materials, POs, inventory | Genie + Metric View |
| **Logistics Tool** | Genie space for carriers, shipments, routes, events | Genie + Metric View |
| **Endpoint** | REST API for downstream apps | Model Serving |
| **Access Control** | Per-user permissions on sub-agents | Built-in |

**Key difference from custom agents:** No manual routing logic needed. AgentBricks uses the tool descriptions + supervisor instructions to automatically determine which sub-agent(s) to call for each question.